# Exotic-style payoffs: European, Asian, American (LSMC)

**Start here:** This deep dive expands on `07_advanced_quant/monte_carlo_simulation.ipynb`; read the overview first for the common `TimeGrid`, `McEngine`, and pricer workflow.

Use the **high-level pricers** in `finstack_quant.monte_carlo` for **vanilla Europeans**, **path-dependent Asians**, and **American-style** exercise via **Longstaff–Schwartz** Monte Carlo.


## Concept

- **European**: payoff from **terminal** spot only; baseline for volatility and rates exposure.
- **Asian**: depends on an **average** along the simulated path; typically **lower volatility** than a vanilla with the same strike.
- **American**: **early exercise** optimizes stopping; **`LsmcPricer`** regresses continuation values on basis functions (here **Laguerre**, wired in Rust).

All examples below use **GBM** and the same seed for apples-to-apples noise (still **different payoffs**, so levels are not directly comparable beyond intuition).

**LSMC note:** `num_steps` sets the Bermudan exercise grid; a **sparse** grid can **bias** the estimated early-exercise boundary (often **understating** American value vs finer grids). The examples use **252** steps as a practical default; increase paths/steps if validation against Europeans looks unstable.

## API walkthrough

`EuropeanPricer.price_call`, `PathDependentPricer.price_asian_call`, and `LsmcPricer.price_american_put` return **`MoneyEstimate`** with **`mean`**, **`stderr`**, and **`ci_lower` / `ci_upper`**.

In [1]:
from finstack_quant.monte_carlo import (
    EuropeanPricer,
    PathDependentPricer,
    LsmcPricer,
    black_scholes_call,
    black_scholes_put,
)

pricer = EuropeanPricer(num_paths=20_000, seed=42)
call = pricer.price_call(spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0)
print(f"European MC call mean: {call.mean.amount:.6f} stderr={call.stderr:.6f}")

asian = PathDependentPricer(num_paths=20_000, seed=42)
asian_call = asian.price_asian_call(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.20, expiry=1.0, num_steps=252
)
print(f"Asian MC call mean:    {asian_call.mean.amount:.6f} stderr={asian_call.stderr:.6f}")

lsmc = LsmcPricer(num_paths=20_000, seed=42)
am_put = lsmc.price_american_put(
    spot=100.0, strike=100.0, rate=0.05, div_yield=0.0, vol=0.30, expiry=1.0, num_steps=252
)
print(f"American MC put mean:  {am_put.mean.amount:.6f} stderr={am_put.stderr:.6f}")

bs_c = black_scholes_call(100.0, 100.0, 0.05, 0.0, 0.20, 1.0)
bs_p = black_scholes_put(100.0, 100.0, 0.05, 0.0, 0.30, 1.0)
print(f"BS European call (vol 20%): {bs_c:.6f}")
print(f"BS European put  (vol 30%): {bs_p:.6f}")


European MC call mean: 10.568528 stderr=0.104628
Asian MC call mean:    5.813236 stderr=0.056836


American MC put mean:  9.932422 stderr=0.078228
BS European call (vol 20%): 10.450584
BS European put  (vol 30%): 9.354197


## Examples

Side-by-side **95% confidence intervals** (module exposes **`ci_lower` / `ci_upper`** on **`MoneyEstimate`**).

In [2]:
from finstack_quant.monte_carlo import EuropeanPricer, PathDependentPricer, LsmcPricer

eu = EuropeanPricer(num_paths=20_000, seed=42).price_call(
    100.0, 100.0, 0.05, 0.0, 0.20, 1.0
)
as_ = PathDependentPricer(num_paths=20_000, seed=42).price_asian_call(
    100.0, 100.0, 0.05, 0.0, 0.20, 1.0, num_steps=252
)
am = LsmcPricer(num_paths=20_000, seed=42).price_american_put(
    100.0, 100.0, 0.05, 0.0, 0.30, 1.0, num_steps=252
)

rows = [
    ("European call (GBM MC)", eu),
    ("Asian call arithmetic (GBM MC)", as_),
    ("American put LSMC (GBM)", am),
]
print("contract | mean | 95% CI low | 95% CI high | stderr")
for name, r in rows:
    print(
        f"{name} | {r.mean.amount:.6f} | {r.ci_lower.amount:.6f} | {r.ci_upper.amount:.6f} | {r.stderr:.6f}"
    )


contract | mean | 95% CI low | 95% CI high | stderr
European call (GBM MC) | 10.568528 | 10.363461 | 10.773594 | 0.104628
Asian call arithmetic (GBM MC) | 5.813236 | 5.701839 | 5.924632 | 0.056836
American put LSMC (GBM) | 9.932422 | 9.779099 | 10.085745 | 0.078228


## Estimates in depth

`MoneyEstimate` exposes additional statistics when captured by the engine (percentiles, min/max etc. may be None unless the underlying run asked for them).


In [3]:
from finstack_quant.monte_carlo import EuropeanPricer
r = EuropeanPricer(num_paths=5000, seed=123).price_call(100, 100, 0.05, 0.0, 0.20, 1.0)
print("mean:", r.mean)
print("stderr:", r.stderr, "relative_stderr():", r.relative_stderr())
print("num_paths / simulated:", r.num_paths, r.num_simulated_paths)
print("ci:", r.ci_lower, r.ci_upper)
print("std_dev / median / p25 / p75 / min / max:", r.std_dev, r.median, r.percentile_25, r.percentile_75, r.min, r.max)

# Quick demo of additional payoff variants (put sides)
ap = PathDependentPricer(num_paths=2000, seed=42).price_asian_put(100,100,0.05,0,0.2,1.0, num_steps=64)
print("asian put mean:", ap.mean.amount)


mean: USD 10.39
stderr: 0.20740933422164612 relative_stderr(): 0.01995428935005063
num_paths / simulated: 5000 5000
ci: USD 9.99 USD 10.80
std_dev / median / p25 / p75 / min / max: 14.666054670951302 None None None None None
asian put mean: 3.334518816753132


## Takeaways

- **`EuropeanPricer`** is the fastest mental model: one payoff date, check against **Black–Scholes**.
- **`PathDependentPricer`** targets **streaming path statistics** (here **arithmetic Asian**).
- **`LsmcPricer`** prices **Bermudan/American-style** exercise on a discrete grid; **basis choice** and **`num_steps`** matter for exercise boundary quality (coarse grids introduce **exercise grid bias**).
- Always report **stderr** or **confidence intervals** with Monte Carlo **levels**.